# Enfermedad cardiaca: factores de riesgo y modelos predictivos (CRISP-DM)

**Autor:** Alejandro Pedreros
**Fecha:** 22/11/2026

Este cuadernillo estudia qué factores de salud y hábitos se asocian con infarto o enfermedad cardiaca en adultos de Estados Unidos. Usa la encuesta BRFSS 2015 de los CDC, con 253.680 registros. Con esos datos entrena y evalúa modelos que estiman el riesgo, para apoyar la prevención.

Fuente: *Heart Disease Health Indicators* (alexteboul), Kaggle, licencia CC0.

## Librerías

Todas las librerías que usa el cuadernillo se importan en la celda siguiente.

In [ ]:
import os
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay,
                             average_precision_score, brier_score_loss, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedKFold, cross_validate,
                                     learning_curve, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

%matplotlib inline

## Configuración y funciones auxiliares

Define las rutas, los nombres de variables y las funciones que dan formato a las tablas.

In [ ]:
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
pd.set_option("display.max_columns", None)

URL_DATOS = ("https://raw.githubusercontent.com/Angher-25/enfermedad-cardiaca-ciencia-de-datos"
             "/main/data/heart_disease_health_indicators_BRFSS2015.csv")
SEMILLA = 42
TARGET = "HeartDiseaseorAttack"
NUMERICAS = ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Education", "Income"]
BINARIAS = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "Diabetes",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
            "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"]
FEATURES = ["HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke", "Diabetes",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
            "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex",
            "Age", "Education", "Income"]

Path("data").mkdir(exist_ok=True)
Path("reports/figures").mkdir(parents=True, exist_ok=True)

def _formato_p(valor):
    if pd.isna(valor):
        return "-"
    return "< 0,0001" if valor < 0.0001 else f"{valor:.4f}"

def _formato_dict(valor):
    if isinstance(valor, dict):
        return "  ·  ".join(f"{k} → {v * 100:.2f} %" for k, v in valor.items())
    return valor

def mostrar(tabla, titulo=None):
    if hasattr(tabla, "to_frame"):
        tabla = tabla.to_frame()
    tabla = tabla.copy()
    formatos = {}
    for col in tabla.columns:
        if "p_valor" in str(col):
            formatos[col] = _formato_p
        elif pd.api.types.is_float_dtype(tabla[col]) and tabla[col].dropna().apply(float.is_integer).all():
            formatos[col] = "{:,.0f}"
        elif tabla[col].apply(lambda v: isinstance(v, dict)).any():
            tabla[col] = tabla[col].apply(_formato_dict)
    numericas = [c for c in tabla.select_dtypes("number").columns if "p_valor" not in str(c)]
    estilo = (tabla.style
              .format(formatos, precision=4, na_rep="-")
              .set_properties(**{"text-align": "center", "padding": "6px 12px",
                                 "color": "#1a1a1a", "background-color": "#ffffff"})
              .set_table_styles([
                  {"selector": "table", "props": [("border-collapse", "collapse")]},
                  {"selector": "th", "props": [("background-color", "#2a7ab0"), ("color", "white"),
                                                ("font-weight", "bold"), ("text-align", "center"),
                                                ("padding", "8px 12px")]},
                  {"selector": "td", "props": [("color", "#1a1a1a"),
                                                ("border", "1px solid #9fb8c8")]},
                  {"selector": "tr:nth-child(even)", "props": [("background-color", "#f4f8fb")]},
                  {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "1.1em"),
                                                     ("font-weight", "bold"), ("color", "#2a7ab0"),
                                                     ("text-align", "left"), ("padding-bottom", "8px")]},
              ]))
    if numericas:
        estilo = estilo.bar(subset=numericas, color="#9ecae1", vmin=0)
    if titulo:
        estilo = estilo.set_caption(titulo)
    display(estilo)

## 1. Datos crudos

Antes de cualquier limpieza se muestra la base tal como se descargó desde GitHub: primeras filas, dimensiones, tipos de dato y valores nulos.

In [ ]:
df_raw = pd.read_csv(URL_DATOS)
display(df_raw.head())
print("Filas y columnas:", df_raw.shape)
mostrar(pd.DataFrame({"tipo": df_raw.dtypes.astype(str),
                      "nulos": df_raw.isna().sum(),
                      "valores_unicos": df_raw.nunique()}), titulo="Tipos de dato y valores nulos (datos crudos)")
df_raw.info()

Qué significa: la base tiene **253.680 filas y 22 columnas**, todas numéricas. Son binarias (0/1), ordinales con códigos de la encuesta (por ejemplo, `Age` de 1 a 13 o `GenHlth` de 1 a 5) y dos continuas (`BMI`, y los días de mala salud `MentHlth` y `PhysHlth`). No hay valores nulos. Lo que sí hay que revisar son los duplicados y los rangos de cada código, que se hace en la sección siguiente.

## 2. Limpieza y preparación de los datos

### 2.1 Valores nulos y duplicados

Se cuenta cuántos valores faltan y cuántas filas están repetidas. Las filas repetidas pueden sesgar las estadísticas y los modelos.

In [ ]:
nulos_totales = int(df_raw.isna().sum().sum())
duplicados = int(df_raw.duplicated().sum())
print("Nulos totales:", nulos_totales)
print("Filas duplicadas exactas:", duplicados)

Qué significa: no hay nulos, pero **23.899 filas son duplicados exactos** (casi 1 de cada 10). Son respuestas idénticas en las 22 columnas, probablemente registros repetidos al cargar la encuesta. Si se dejan, el mismo perfil cuenta varias veces y las cifras quedan infladas.

### 2.2 Eliminación de duplicados

Se conservan las filas únicas. Se compara el balance del evento antes y después para confirmar que el cambio no altera el problema.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Filas originales: {len(df_raw)}  |  sin duplicados: {len(df)}")
mostrar(pd.DataFrame({
    "con_duplicados": df_raw[TARGET].value_counts(normalize=True).round(4),
    "sin_duplicados": df[TARGET].value_counts(normalize=True).round(4),
}), titulo="Proporción del evento (infarto o enfermedad cardiaca)")

Qué significa: quedan **229.781 registros**. La proporción del evento sube de 9,4 % a **10,3 %**, porque los duplicados eran sobre todo personas sin evento. El problema sigue siendo desbalanceado: hay muchos más sin evento que con evento, y por eso más adelante se usa `class_weight="balanced"`.

### 2.3 Validación de rangos según el libro de códigos BRFSS

Cada variable tiene un rango válido. Se verifica que ningún valor quede fuera y que las binarias solo tengan 0 y 1.

In [ ]:
RANGOS = {
    "BMI": (12, 98), "MentHlth": (0, 30), "PhysHlth": (0, 30),
    "GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8),
    "Diabetes": (0, 2),
}
filas_validacion = []
for col, (lo, hi) in RANGOS.items():
    fuera = int(((df[col] < lo) | (df[col] > hi)).sum())
    filas_validacion.append({"variable": col, "rango_esperado": f"{lo} a {hi}",
                             "valores_fuera": fuera, "resultado": "OK" if fuera == 0 else "revisar"})
for col in BINARIAS:
    valores = sorted(df[col].dropna().unique().tolist())
    ok = set(valores) <= {0, 1}
    filas_validacion.append({"variable": col, "rango_esperado": "0 o 1",
                             "valores_fuera": int((~df[col].isin([0, 1])).sum()),
                             "resultado": "OK" if ok else "revisar"})
tabla_validacion = pd.DataFrame(filas_validacion)
mostrar(tabla_validacion, titulo="Validación de rangos por variable")
print("Variables con problemas:", int((tabla_validacion["resultado"] != "OK").sum()))

Qué significa: todas las variables están dentro de su rango. Esto confirma que los códigos se leen bien y que no hay errores de digitación que haya que corregir o eliminar. Si alguna fallara, se revisaría antes de seguir.

### 2.4 Variables derivadas

Se crean variables más fáciles de interpretar: edad aproximada en años, categoría de IMC, número de condiciones de salud, inactividad física y sexo como texto.

In [ ]:
EDAD_APROX = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57,
              9: 62, 10: 67, 11: 72, 12: 77, 13: 82}
df["edad_aprox"] = df["Age"].map(EDAD_APROX)
df["categoria_imc"] = pd.cut(df["BMI"], bins=[0, 18.5, 25, 30, 1000],
                             labels=["bajo_peso", "normal", "sobrepeso", "obesidad"],
                             right=False).astype(str)
df["num_condiciones"] = df[["HighBP", "HighChol", "Diabetes", "Smoker", "Stroke"]].gt(0).sum(axis=1)
df["inactivo"] = (1 - df["PhysActivity"]).astype(int)
df["sexo"] = df["Sex"].map({0: "Mujer", 1: "Hombre"})
display(df[["Age", "edad_aprox", "BMI", "categoria_imc", "num_condiciones", "inactivo", "sexo"]].head())
mostrar(df["categoria_imc"].value_counts(normalize=True).rename("proporcion"),
        titulo="Distribución de la categoría de IMC")

Qué significa: la edad aproximada convierte el código 1 a 13 en años (por ejemplo, el grupo 13 corresponde a 80 años o más). La categoría de IMC y el número de condiciones resumen factores de riesgo en una sola variable. Estas variables se usan para la base de datos; los modelos usan las 21 variables originales de la lista `FEATURES`.

### 2.5 Carga en base de datos SQLite

Se guardan los datos crudos y los limpios en una base SQLite. Así las consultas posteriores no dependen del CSV.

In [ ]:
BD = Path("data/heart_disease.db")
if BD.exists():
    BD.unlink()
with sqlite3.connect(BD) as con:
    df_raw.to_sql("raw_encuesta", con, index=False)
    df.to_sql("encuestados", con, index=False)
    con.execute("CREATE INDEX idx_enc_target ON encuestados(HeartDiseaseorAttack)")
    con.execute("CREATE INDEX idx_enc_edad ON encuestados(Age)")
    con.commit()
with sqlite3.connect(BD) as con:
    n_bd = con.execute("SELECT COUNT(*) FROM encuestados").fetchone()[0]
    tasas_edad = pd.read_sql(
        "SELECT Age AS grupo_edad_brfss, ROUND(AVG(HeartDiseaseorAttack), 4) AS tasa_evento, "
        "COUNT(*) AS n FROM encuestados GROUP BY Age ORDER BY Age", con)
print(f"Base creada en {BD} con {n_bd} filas en 'encuestados'.")
mostrar(tasas_edad, titulo="Tasa de evento por grupo de edad (consulta SQL)")

Qué significa: la base tiene dos tablas. `raw_encuesta` conserva todo lo original; `encuestados` tiene los datos sin duplicados y con variables derivadas. La tabla de tasas por edad es la primera señal de que la edad es un factor importante: la tasa sube con cada grupo.

## 3. Comprensión de los datos (análisis exploratorio)

Con los datos ya limpios se describen las variables y se mide cuáles se asocian con el evento.

### 3.1 Estadísticas descriptivas

Resumen de cada variable numérica y binaria: media, desviación, mínimo, cuartiles y máximo.

In [ ]:
resumen = df[NUMERICAS + BINARIAS].describe().T.round(2)
resumen.to_csv("reports/estadisticas_descriptivas.csv")
mostrar(resumen, titulo="Estadísticas descriptivas (sin duplicados)")

Qué significa: `MentHlth` y `PhysHlth` tienen medias bajas pero máximos de 30, así que la mayoría responde 0 días de mala salud y unos pocos reportan mucho. `Age` y `GenHlth` son códigos ordinales: sus medias no son edades ni notas, solo ayudan a ver el orden general.

### 3.2 Variables numéricas frente al evento (Mann-Whitney U)

La prueba compara la distribución de cada variable numérica entre quienes tienen el evento y quienes no. Se usa una prueba no paramétrica porque las variables tienen colas largas.

In [ ]:
filas = []
for col in NUMERICAS:
    a = df.loc[df[TARGET] == 0, col]
    b = df.loc[df[TARGET] == 1, col]
    _, p = mannwhitneyu(a, b)
    filas.append({"variable": col,
                  "media_sin_evento": round(a.mean(), 2),
                  "media_con_evento": round(b.mean(), 2),
                  "diferencia_std": round((b.mean() - a.mean()) / df[col].std(), 4),
                  "p_valor": p})
tabla_num = pd.DataFrame(filas)
tabla_num.to_csv("reports/asociacion_numericas_con_riesgo.csv", index=False)
mostrar(tabla_num, titulo="Variables numéricas frente al evento")

Qué significa: en todas las variables numéricas la diferencia entre grupos es significativa (p < 0,0001). Quienes tienen el evento son en promedio **7,9 vs 10,1** en `Age` (código de edad) y **4,2 vs 9,2** en días de mala salud física. La diferencia estandarizada más alta aparece en `GenHlth` y `Age`, así que son las variables que más separan a los dos grupos.

### 3.3 Variables binarias frente al evento (chi-cuadrado)

Para cada variable binaria se calcula la tasa de evento en cada nivel y la prueba chi-cuadrado de independencia.

In [ ]:
filas = []
for col in BINARIAS:
    tabla = pd.crosstab(df[col], df[TARGET])
    _, p, _, _ = chi2_contingency(tabla)
    tasas = df.groupby(col)[TARGET].mean().round(4).to_dict()
    filas.append({"variable": col, "tasa_evento_por_nivel": tasas, "p_valor": p})
tabla_cat = pd.DataFrame(filas)
tabla_cat.to_csv("reports/asociacion_binarias_con_riesgo.csv", index=False)
mostrar(tabla_cat, titulo="Variables binarias frente al evento")

Qué significa: la tasa de evento es **varias veces más alta** cuando la condición está presente. Por ejemplo, con presión alta la tasa pasa de 4,7 % a 17,0 %, y con antecedente de accidente cerebrovascular pasa de 9,0 % a 38,3 %. `Fruits` es la única variable con un efecto muy pequeño, aunque su p-valor sigue siendo significativo.

### 3.4 Correlación con el evento

Correlación de Pearson de cada variable con el objetivo. Un valor cercano a 0 indica poca relación lineal.

In [ ]:
correlacion = df[NUMERICAS + BINARIAS + [TARGET]].corr()[TARGET].drop(TARGET)
correlacion = correlacion.sort_values()
correlacion.to_csv("reports/correlacion_con_target.csv", header=["r"])
mostrar(correlacion.rename("r"), titulo="Correlación de cada variable con el evento")

Qué significa: las correlaciones son moderadas en magnitud. Las más altas en valor absoluto corresponden a edad, salud general y dificultad para caminar. Una correlación baja no significa que una variable no sirva, porque los modelos pueden captar relaciones no lineales que la correlación no mide.

### Figura 1: distribución de las variables numéricas

Histogramas de las variables numéricas y de `Smoker`, con los datos sin duplicados.

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(12, 6))
for eje, col in zip(ejes.flat, NUMERICAS + ["Smoker"]):
    sns.histplot(df[col], bins=30, ax=eje, color="#2a7ab0")
    eje.set_title(col)
fig.tight_layout()
fig.savefig("reports/figures/01_distribuciones.png", dpi=110)
plt.show()

Qué significa para el negocio: la mayoría de las personas tiene índice de masa corporal entre 22 y 32, y la mayor parte reporta cero días de mala salud. Esto indica que una minoría con muchos días de mala salud concentra buena parte del riesgo de atención. `GenHlth` se concentra en 2 y 3, lo que muestra que la mayoría se percibe en buen estado general.

### Figura 2: matriz de correlación

Mapa de calor con la correlación entre todas las variables y el objetivo.

In [ ]:
fig, eje = plt.subplots(figsize=(7.5, 7.5))
sns.heatmap(df[NUMERICAS + BINARIAS + [TARGET]].corr(), cmap="RdBu_r",
            center=0, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=eje)
eje.set_title("Matriz de correlación (sin duplicados)")
fig.tight_layout()
fig.savefig("reports/figures/02_correlacion.png", dpi=110)
plt.show()

Qué significa para el negocio: la presión alta y el colesterol alto aparecen juntos, y lo mismo pasa con la salud general y la dificultad para caminar. En la práctica, una persona con varias condiciones suele tener varias de estas señales a la vez. Por eso el modelo debe considerar las variables en conjunto y no una por una.

### Figura 3: tasa de evento por edad y por salud general

Cada barra es la proporción de personas con el evento dentro de un grupo.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.barplot(data=df.groupby("Age")[TARGET].mean().reset_index(),
            x="Age", y=TARGET, ax=ejes[0], color="#c0392b")
ejes[0].set_title("Tasa de evento por grupo de edad (código BRFSS)")
sns.barplot(data=df.groupby("GenHlth")[TARGET].mean().reset_index(),
            x="GenHlth", y=TARGET, ax=ejes[1], color="#8e44ad")
ejes[1].set_title("Tasa de evento por salud general (1 = excelente, 5 = mala)")
fig.tight_layout()
fig.savefig("reports/figures/03_tasas_edad_salud.png", dpi=110)
plt.show()

Qué significa para el negocio: la tasa sube de unos 0,5 % en los grupos más jóvenes a cerca de 25 % en los mayores de 80 años. Quienes califican su salud como mala tienen una tasa varias veces mayor que quienes la califican como excelente. Esto sugiere que la edad y la percepción de salud son señales de alerta que un programa de prevención puede usar para priorizar a personas.

## 4. Modelamiento y evaluación (fases 4 y 5 de CRISP-DM)

Se entrenan y comparan cuatro modelos. La validación cruzada se hace solo sobre el entrenamiento. La evaluación final se hace sobre un conjunto de prueba que el modelo no vio.

### 4.1 Partición de datos

El 80 % se usa para entrenar y el 20 % para evaluar. La partición es estratificada, así que ambas partes tienen la misma proporción de evento.

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
print(f"Entrenamiento: {len(X_tr)}  |  prueba: {len(X_te)}  |  "
      f"prevalencia en prueba: {y_te.mean():.4f}")

Qué significa: el conjunto de prueba tiene casi 46.000 registros. Como tiene la misma prevalencia que el total (10,3 %), las métricas de prueba son representativas de la población de la encuesta.

### 4.2 Modelos

- **Línea base:** siempre predice la clase mayoritaria. Sirve para medir cuánto aporta un modelo real.
- **Regresión logística:** requiere escalado (`StandardScaler`) porque penaliza por igual todas las variables.
- **Random Forest y gradient boosting:** basados en árboles, no requieren escalado.
- `class_weight` compensa el desbalance del evento.

In [ ]:
modelos = {
    "Línea base (mayoritaria)": make_pipeline(
        StandardScaler(), DummyClassifier(strategy="most_frequent")),
    "Regresión logística": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=50,
        class_weight="balanced_subsample", n_jobs=4, random_state=SEMILLA),
    "Gradient boosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=SEMILLA),
}
print("Modelos definidos:", ", ".join(modelos))

Qué significa: la regresión logística es fácil de explicar a un equipo de salud. El Random Forest y el gradient boosting pueden captar relaciones más complejas. Los tres se comparan con la línea base para confirmar que realmente aprenden algo.

### 4.3 Validación cruzada (solo entrenamiento, 5 folds)

El entrenamiento se divide en 5 partes. Cada modelo se entrena 5 veces y se evalúa en la parte que dejó fuera.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
filas = []
for nombre, modelo in modelos.items():
    r = cross_validate(modelo, X_tr, y_tr, cv=cv,
                       scoring=["roc_auc", "average_precision", "f1"], n_jobs=1)
    filas.append({
        "modelo": nombre,
        "cv_roc_auc": round(r["test_roc_auc"].mean(), 4),
        "cv_roc_auc_std": round(r["test_roc_auc"].std(), 4),
        "cv_pr_auc": round(r["test_average_precision"].mean(), 4),
        "cv_f1": round(r["test_f1"].mean(), 4),
    })
tabla_cv = pd.DataFrame(filas)
tabla_cv.to_csv("reports/metricas_validacion_cruzada.csv", index=False)
mostrar(tabla_cv, titulo="Validación cruzada (5 folds, entrenamiento)")

Qué significa: los tres modelos reales obtienen un ROC-AUC de **0,836 a 0,838** en validación cruzada, con variaciones de 0,003. Esa estabilidad indica que el resultado no depende de una partición afortunada. La línea base tiene ROC-AUC de 0,50, es decir, no distingue a nadie.

### 4.4 Evaluación en el conjunto de prueba

Cada modelo se entrena con todo el entrenamiento y se evalúa en el 20 % reservado. Las métricas de umbral usan 0,5 como punto de corte.

In [ ]:
filas = []
probs = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    prob = modelo.predict_proba(X_te)[:, 1]
    probs[nombre] = prob
    pred = (prob >= 0.5).astype(int)
    filas.append({
        "modelo": nombre,
        "roc_auc": round(roc_auc_score(y_te, prob), 4),
        "pr_auc": round(average_precision_score(y_te, prob), 4),
        "precision_umbral_0.5": round(precision_score(y_te, pred, zero_division=0), 4),
        "recall_umbral_0.5": round(recall_score(y_te, pred, zero_division=0), 4),
        "f1_umbral_0.5": round(f1_score(y_te, pred, zero_division=0), 4),
    })
tabla_test = pd.DataFrame(filas)
tabla_test.to_csv("reports/metricas_test.csv", index=False)
mostrar(tabla_test, titulo="Métricas en el conjunto de prueba (20 %)")

Qué significa para el negocio: con umbral 0,5, la regresión logística detecta **cerca de 78 %** de las personas que realmente tendrán el evento, y el gradient boosting detecta **80 %**. La **precisión** ronda 25 %: de cada cuatro alertas, tres son falsas. Es un tamizaje útil para ordenar a quién revisar primero, pero no para diagnosticar. Si se quiere menos falsas alarmas, el umbral puede subir, a costa de detectar menos casos.

### Figura 4: curvas ROC en el conjunto de prueba

La curva ROC muestra la relación entre casos detectados y falsas alarmas para cada umbral. Cuanto más arriba y a la izquierda, mejor. La diagonal es la línea base de un modelo que adivina al azar.

In [ ]:
fig, eje = plt.subplots(figsize=(6, 5))
for nombre, prob in probs.items():
    RocCurveDisplay.from_predictions(y_te, prob, name=nombre, ax=eje)
eje.plot([0, 1], [0, 1], "k--", lw=0.8)
eje.set_title("Curvas ROC en prueba")
fig.tight_layout()
fig.savefig("reports/figures/04_curvas_roc.png", dpi=110)
plt.show()

Qué significa para el negocio: los tres modelos quedan prácticamente encima uno del otro, con AUC de 0,84. Esto indica que la información de la encuesta permite separar bien a las personas con riesgo, y que la elección del modelo importa menos que la calidad de los datos.

### Figura 5: matriz de confusión del mejor modelo (umbral 0,5)

Cruza lo que predijo el modelo con lo que ocurrió. El mejor modelo se elige por ROC-AUC en prueba.

In [ ]:
mejor = tabla_test[tabla_test["modelo"] != "Línea base (mayoritaria)"] \
    .sort_values("roc_auc", ascending=False).iloc[0]["modelo"]
pred_mejor = (probs[mejor] >= 0.5).astype(int)
fig, eje = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_te, pred_mejor, ax=eje, cmap="Blues", values_format="d",
                                        display_labels=["Sin evento", "Con evento"])
eje.set_title(f"Matriz de confusión (umbral 0,5)\n{mejor}")
fig.tight_layout()
fig.savefig("reports/figures/05_matriz_confusion.png", dpi=110)
plt.show()

Qué significa para el negocio: en la esquina inferior derecha están las personas con evento que el modelo detecta. En la inferior izquierda están los casos que se escapan. En la superior derecha, las falsas alarmas. Para un tamizaje, es preferible tener algunas falsas alarmas que dejar pasar casos reales, pero el umbral debe decidirlo el equipo de salud según el costo de cada error.

### 4.5 Ganancia por deciles (Random Forest)

Las personas de prueba se ordenan de mayor a menor riesgo predicho y se agrupan en 10 partes. Se mide cuántos casos reales hay en cada parte.

In [ ]:
decil_df = pd.DataFrame({"prob": probs["Random Forest"], "y": y_te.to_numpy()})
decil_df["decil"] = pd.qcut(decil_df["prob"].rank(method="first", ascending=False),
                            10, labels=range(1, 11))
ganancia = decil_df.groupby("decil", observed=True)["y"].agg(["mean", "size"]) \
    .rename(columns={"mean": "tasa_evento", "size": "n"})
ganancia["lift"] = ganancia["tasa_evento"] / y_te.mean()
ganancia = ganancia.round(4)
ganancia.to_csv("reports/ganancia_por_decil_random_forest.csv")
mostrar(ganancia, titulo="Tasa de evento por decil de riesgo (1 = mayor riesgo predicho)")

Qué significa para el negocio: el primer decil (el 10 % de personas con mayor riesgo predicho) tiene una tasa de evento de **40 %**, es decir, unas 3,9 veces el promedio. El último decil tiene una tasa cercana a cero. Esto permite priorizar a quién contactar primero en una campaña de prevención, con un alcance limitado a los que más lo necesitan.

### Figura 6: ganancia por deciles

La misma información de la tabla anterior, en barras.

In [ ]:
fig, eje = plt.subplots(figsize=(7, 4))
eje.bar(ganancia.index.astype(int), ganancia["tasa_evento"], color="#c0392b")
eje.axhline(y_te.mean(), color="k", ls="--", lw=0.8, label="tasa media")
eje.set_xlabel("Decil de riesgo predicho (1 = mayor)")
eje.set_ylabel("Tasa real de evento")
eje.set_title("Ganancia por deciles (Random Forest, prueba)")
eje.legend()
fig.tight_layout()
fig.savefig("reports/figures/06_ganancia_por_decil.png", dpi=110)
plt.show()

Qué significa para el negocio: la barra del primer decil supera con holgura la línea de la tasa media, y las barras bajan de forma constante. Un programa que contacte solo al 20 % de mayor riesgo captaría alrededor de 62 % de los casos reales, según estos resultados.

### 4.6 Curvas de aprendizaje

Compara el desempeño en entrenamiento y en validación al crecer el tamaño de los datos. Una brecha grande indica sobreajuste.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for eje, nombre in zip(ejes, ["Regresión logística", "Random Forest"]):
    tam, tr_sc, va_sc = learning_curve(
        modelos[nombre], X_tr, y_tr, cv=3, scoring="average_precision",
        train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=1, random_state=SEMILLA)
    eje.plot(tam, tr_sc.mean(axis=1), "o-", label="entrenamiento")
    eje.plot(tam, va_sc.mean(axis=1), "o-", label="validación")
    eje.set_title(f"Curva de aprendizaje: {nombre}")
    eje.set_xlabel("Tamaño de entrenamiento")
    eje.set_ylabel("PR-AUC")
    eje.legend()
fig.tight_layout()
fig.savefig("reports/figures/07_curvas_aprendizaje.png", dpi=110)
plt.show()

Qué significa para el negocio: las dos curvas de validación se estabilizan con pocos datos, lo que indica que más registros no cambiarían mucho el resultado. El Random Forest tiene una brecha mayor entre entrenamiento y validación que la regresión logística, así que memoriza un poco más.

### 4.7 Explicabilidad con SHAP (Random Forest)

SHAP asigna a cada variable una contribución positiva o negativa al riesgo de cada persona. Se usa una muestra de 300 registros de prueba por costo computacional.

In [ ]:
rf = modelos["Random Forest"]
muestra = X_te.sample(300, random_state=SEMILLA)
explicador = shap.TreeExplainer(rf)
valores = explicador.shap_values(muestra, check_additivity=False)
if isinstance(valores, list):
    valores = valores[1]
elif valores.ndim == 3:
    valores = valores[:, :, 1]

fig = plt.figure()
shap.summary_plot(valores, muestra, show=False, max_display=15)
plt.tight_layout()
plt.savefig("reports/figures/08_shap_resumen.png", dpi=110, bbox_inches="tight")
plt.show()

Qué significa para el negocio: cada punto es una persona. Los puntos a la derecha empujan el riesgo hacia arriba, y el color indica si la variable tiene un valor alto (rojo) o bajo (azul). La edad alta y la presión alta empujan el riesgo hacia arriba. La salud general mala también. Estas son las señales que un personal de salud puede revisar en una consulta.

### 4.8 Importancia SHAP media (tabla)

Promedio del valor absoluto de SHAP por variable. Mide cuánto cambia la predicción cada variable, en promedio.

In [ ]:
importancia_shap = pd.Series(np.abs(valores).mean(axis=0), index=FEATURES) \
    .sort_values(ascending=False)
importancia_shap.to_csv("reports/importancia_shap_random_forest.csv", header=["abs_shap_medio"])
mostrar(importancia_shap.head(10).round(4).rename("abs_shap_medio"),
        titulo="Importancia SHAP media (Random Forest, top 10)")

Qué significa para el negocio: la edad es la variable con mayor peso, seguida de la presión alta, la salud general, el colesterol alto y el sexo.

### 4.9 Importancia por permutación (gradient boosting)

Se desordena una variable a la vez y se mide cuánto cae el PR-AUC. Es una segunda forma de medir importancia, independiente de SHAP.

In [ ]:
gb = modelos["Gradient boosting"]
perm = permutation_importance(gb, X_te, y_te, scoring="average_precision",
                              n_repeats=3, random_state=SEMILLA, n_jobs=1)
importancia_perm = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
importancia_perm.to_csv("reports/importancia_permutacion_gb.csv", header=["reduccion_pr_auc"])
mostrar(importancia_perm.head(10).round(4).rename("reduccion_pr_auc"),
        titulo="Importancia por permutación (gradient boosting, top 10)")

Qué significa para el negocio: la edad, la salud general y el accidente cerebrovascular son las variables cuya ausencia más empeora la predicción. Coincide en lo esencial con SHAP, lo que da más confianza en que las conclusiones no dependen del método.

### 4.10 Coeficientes de la regresión logística

Los coeficientes muestran la dirección y la fuerza de cada variable. Como las variables están estandarizadas, son comparables entre sí.

In [ ]:
logit = modelos["Regresión logística"]
coef = pd.Series(logit.named_steps["logisticregression"].coef_[0], index=FEATURES)
coef = coef.reindex(coef.abs().sort_values(ascending=False).index)
coef.to_csv("reports/coeficientes_regresion_logistica.csv", header=["coeficiente"])
mostrar(coef.head(10).round(4).rename("coeficiente"), titulo="Coeficientes de la regresión logística (top 10)")

Qué significa para el negocio: un coeficiente positivo significa que la variable aumenta el riesgo, y uno negativo que lo reduce, manteniendo lo demás constante. Edad, salud general, sexo, colesterol alto y presión alta son los coeficientes más grandes. Así se puede explicar al equipo de salud por qué el modelo asigna más riesgo a una persona.

## 5. Complemento: robustez, equidad y calibración

Se revisan tres cosas: si las diferencias entre modelos son reales, si el modelo funciona igual en distintos grupos, y si sus probabilidades se pueden leer como riesgo absoluto.

### 5.1 Intervalos de confianza por bootstrap

Se remuestrea el conjunto de prueba 1.000 veces con reemplazo. El intervalo del 95 % va de los percentiles 2,5 y 97,5 de las métricas.

In [ ]:
p = {"Regresión logística": probs["Regresión logística"],
     "Random Forest": probs["Random Forest"]}
yt = y_te.values
rng = np.random.default_rng(SEMILLA)
indices = [rng.integers(0, len(yt), len(yt)) for _ in range(1000)]
filas = []
for nombre, pr in p.items():
    auc_b, pr_b = [], []
    for ii in indices:
        if yt[ii].min() == yt[ii].max():
            continue
        auc_b.append(roc_auc_score(yt[ii], pr[ii]))
        pr_b.append(average_precision_score(yt[ii], pr[ii]))
    for metrica, valor, boot in [("ROC-AUC", roc_auc_score(yt, pr), auc_b),
                                 ("PR-AUC", average_precision_score(yt, pr), pr_b)]:
        lo, hi = np.percentile(boot, [2.5, 97.5])
        filas.append({"modelo": nombre, "metrica": metrica, "valor": round(valor, 4),
                      "ic95_inferior": round(lo, 4), "ic95_superior": round(hi, 4)})
tabla_ic = pd.DataFrame(filas)
tabla_ic.to_csv("reports/intervalos_confianza_bootstrap.csv", index=False)
mostrar(tabla_ic, titulo="Intervalos de confianza al 95 % (bootstrap)")

Qué significa: los intervalos de regresión logística y Random Forest se traslapan casi por completo. No hay evidencia de que un modelo sea mejor que el otro. Para el negocio, conviene el más simple y explicable, que es la regresión logística.

### 5.2 Equidad por sexo y edad

Se calculan el ROC-AUC, el recall y la precisión (umbral 0,5) para cada grupo. Una brecha grande entre grupos indica que el modelo no funciona igual para todos.

In [ ]:
te = X_te.copy()
te["y"] = y_te.values

def metricas_grupo(mascara, pr):
    yy = te.loc[mascara, "y"].values
    pp = pr[mascara.values]
    pred = (pp >= 0.5).astype(int)
    tp = int(((pred == 1) & (yy == 1)).sum())
    fp = int(((pred == 1) & (yy == 0)).sum())
    fn = int(((pred == 0) & (yy == 1)).sum())
    return {
        "n": len(yy),
        "prevalencia": round(yy.mean(), 4),
        "roc_auc": round(roc_auc_score(yy, pp), 4) if len(np.unique(yy)) > 1 else np.nan,
        "recall": round(tp / (tp + fn), 4) if (tp + fn) else np.nan,
        "precision": round(tp / (tp + fp), 4) if (tp + fp) else np.nan,
    }

grupos = {
    "Sexo: hombre (1)": te["Sex"] == 1,
    "Sexo: mujer (0)": te["Sex"] == 0,
    "Edad: 65 años o más (código 10 a 13)": te["Age"] >= 10,
    "Edad: menores de 65 (código 1 a 9)": te["Age"] < 10,
}
filas = []
for nombre, pr in p.items():
    for gnombre, m in grupos.items():
        filas.append({"modelo": nombre, "grupo": gnombre, **metricas_grupo(m, pr)})
tabla_eq = pd.DataFrame(filas)
tabla_eq.to_csv("reports/metricas_por_subgrupo.csv", index=False)
mostrar(tabla_eq, titulo="Métricas por subgrupo (umbral 0,5)")

Qué significa para el negocio: con el mismo umbral, el modelo detecta **más casos en hombres (recall de 0,83 a 0,84) que en mujeres (0,71 a 0,73)**, y **más casos en mayores de 65 (0,87 a 0,88) que en menores de 65 (0,61 a 0,64)**. En mujeres la precisión también es menor. En mayores de 65 el ROC-AUC baja a cerca de 0,76, frente a 0,85 en menores de 65. Es decir, distinguir quién tendrá el evento dentro del grupo de mayores es más difícil. Antes de usar el modelo en una campaña real, hay que revisar estas brechas y decidir si se ajusta el umbral por grupo.

### 5.3 Calibración de las probabilidades

Se comparan las probabilidades predichas con la tasa real de evento. Si el modelo está bien calibrado, una probabilidad de 20 % significa que alrededor de uno de cada cinco casos tendrá el evento.

In [ ]:
filas = []
for nombre, pr in p.items():
    filas.append({"modelo": nombre,
                  "brier": round(brier_score_loss(yt, pr), 4),
                  "prevalencia_media_predicha": round(pr.mean(), 4),
                  "prevalencia_observada": round(yt.mean(), 4)})
tabla_brier = pd.DataFrame(filas)
tabla_brier.to_csv("reports/calibracion_brier.csv", index=False)
mostrar(tabla_brier, titulo="Calibración: Brier y prevalencia predicha frente a observada")

Qué significa para el negocio: el promedio de las probabilidades predichas es de unos **0,36**, mientras que la prevalencia real es de **0,10**. Esto ocurre porque `class_weight="balanced"` sube el peso de los casos con evento. Por eso las probabilidades **no deben leerse como riesgo absoluto**: sirven para ordenar a las personas, pero no para decir que alguien tiene 36 % de probabilidad de infarto.

### Figura 7: curva de calibración

Compara la tasa observada con la probabilidad predicha, por deciles de probabilidad.

In [ ]:
fig, eje = plt.subplots(figsize=(6, 5))
for nombre, pr in p.items():
    deciles = pd.qcut(pr, 10, labels=False, duplicates="drop")
    d = pd.DataFrame({"pred": pr, "obs": yt, "dec": deciles}).groupby("dec").mean()
    eje.plot(d["pred"], d["obs"], marker="o", label=nombre)
eje.plot([0, 1], [0, 1], "k--", lw=1, label="Calibración perfecta")
eje.set_xlabel("Probabilidad predicha (media por decil)")
eje.set_ylabel("Tasa observada")
eje.set_title("Curva de calibración en prueba")
eje.legend()
fig.tight_layout()
fig.savefig("reports/figures/09_calibracion.png", dpi=130)
plt.show()

Qué significa para el negocio: la curva queda por debajo de la diagonal perfecta en casi todo el rango. Confirma que las probabilidades están infladas, y que lo importante es el orden de riesgo y no el número absoluto.

## 6. Conclusión general

*Fuentes: secciones 1 a 5 de este cuadernillo.*

**Negocio.** Los factores asociados al infarto o enfermedad cardiaca son la edad, la salud general, la presión alta, el colesterol alto, el antecedente de accidente cerebrovascular, la dificultad para caminar y el sexo. Un programa de prevención puede usar estas señales para priorizar a quién revisar primero.

**Datos.** La base tenía 23.899 duplicados, que se eliminaron. Todos los códigos quedaron dentro de su rango BRFSS. El evento es poco frecuente (10,3 %), así que el problema está desbalanceado.

**Modelamiento.** Regresión logística, Random Forest y gradient boosting rinden casi igual, con ROC-AUC de 0,84 en prueba. El primer decil de riesgo concentra una tasa de evento de 40 %. Con umbral 0,5, el modelo detecta cerca de 80 % de los casos, con una precisión de 25 %.

**Limitaciones.** Los datos son autoreportados y de corte transversal, así que muestran asociaciones, no causas. El modelo no rinde igual para todos los grupos: detecta menos casos en mujeres y en menores de 65, y su ROC-AUC es menor en mayores de 65. Las probabilidades no están calibradas y no deben leerse como riesgo absoluto. No hubo búsqueda de hiperparámetros ni validación en otra población o en otro año de BRFSS.

**Uso recomendado.** Ordenar a las personas por riesgo relativo para priorizar la revisión clínica. No usar el modelo para decidir seguros, empleo ni acceso a servicios, y no reemplazar el diagnóstico médico.

## 7. Reporte ejecutivo

La celda siguiente arma el reporte en `reports/reporte_ejecutivo.md` con las cifras calculadas en este cuadernillo. Si se ejecuta de nuevo, el archivo se reconstruye igual, sin intervención manual.

In [ ]:
def fmt(x, d=4):
    return f"{x:.{d}f}".replace(".", ",")

def pct(x):
    return f"{x * 100:.1f}".replace(".", ",") + " %"

fila_lr = tabla_test.set_index("modelo").loc["Regresión logística"]
fila_rf = tabla_test.set_index("modelo").loc["Random Forest"]
fila_gb = tabla_test.set_index("modelo").loc["Gradient boosting"]
eq = tabla_eq.set_index(["modelo", "grupo"])
rf_mujer = eq.loc[("Random Forest", "Sexo: mujer (0)")]
rf_mayor = eq.loc[("Random Forest", "Edad: 65 años o más (código 10 a 13)")]
rf_menor = eq.loc[("Random Forest", "Edad: menores de 65 (código 1 a 9)")]
brier = tabla_brier.set_index("modelo")
top_shap = importancia_shap.head(5).index.tolist()
decil1 = ganancia.loc[1]
archivos_csv = sorted(p_.name for p_ in Path("reports").glob("*.csv"))
archivos_fig = sorted(p_.name for p_ in Path("reports/figures").glob("*.png"))

reporte = f"""# Reporte ejecutivo: factores de riesgo de enfermedad cardiaca (BRFSS 2015)

## 1. Contextualización y objetivo

La enfermedad cardiaca es una de las principales causas de muerte. Este análisis usa la encuesta BRFSS 2015 de los CDC, con {len(df_raw):,} registros de adultos de Estados Unidos. El objetivo es identificar qué factores de salud y hábitos se asocian con infarto o enfermedad cardiaca, y estimar qué tan bien se puede ordenar el riesgo para apoyar la prevención.

## 2. Metodología CRISP-DM

1. **Comprensión del negocio:** definición de la pregunta y de los límites del estudio.
2. **Comprensión de los datos:** perfil de tipos, nulos y duplicados; pruebas estadísticas.
3. **Preparación:** eliminación de {len(df_raw) - len(df):,} duplicados, validación de rangos BRFSS, variables derivadas y base SQLite.
4. **Modelamiento:** línea base, regresión logística, Random Forest y gradient boosting, con validación cruzada de 5 folds.
5. **Evaluación:** métricas en prueba (20 %), ganancia por deciles, SHAP, importancia por permutación, intervalos bootstrap, equidad y calibración.

## 3. Resultados (hallazgos clave del EDA)

- De {len(df_raw):,} registros, quedaron {len(df):,} tras eliminar duplicados. El evento ocurre en {pct(df[TARGET].mean())} de los encuestados.
- La tasa de evento sube con la edad: de unos 0,5 % en los grupos jóvenes a cerca de 25 % en los mayores de 80 años.
- La tasa de evento pasa de {pct(tasas_edad["tasa_evento"].iloc[0])} en el grupo de edad más joven a {pct(tasas_edad["tasa_evento"].iloc[-1])} en el grupo de 80 años o más.
- Las variables más asociadas al evento son la edad, la salud general, la presión alta, el colesterol alto y el accidente cerebrovascular previo.

## 4. Modelamiento (modelos usados y métricas principales)

Se compararon cuatro modelos. En prueba (20 %):

| Modelo | ROC-AUC | PR-AUC | Precisión (0,5) | Recall (0,5) | F1 (0,5) |
|---|---|---|---|---|---|
| Regresión logística | {fmt(fila_lr["roc_auc"])} | {fmt(fila_lr["pr_auc"])} | {fmt(fila_lr["precision_umbral_0.5"])} | {fmt(fila_lr["recall_umbral_0.5"])} | {fmt(fila_lr["f1_umbral_0.5"])} |
| Random Forest | {fmt(fila_rf["roc_auc"])} | {fmt(fila_rf["pr_auc"])} | {fmt(fila_rf["precision_umbral_0.5"])} | {fmt(fila_rf["recall_umbral_0.5"])} | {fmt(fila_rf["f1_umbral_0.5"])} |
| Gradient boosting | {fmt(fila_gb["roc_auc"])} | {fmt(fila_gb["pr_auc"])} | {fmt(fila_gb["precision_umbral_0.5"])} | {fmt(fila_gb["recall_umbral_0.5"])} | {fmt(fila_gb["f1_umbral_0.5"])} |

- El primer decil de riesgo (Random Forest) tiene una tasa de evento de {pct(decil1["tasa_evento"])}, unas {fmt(decil1["lift"], 1)} veces el promedio.
- Las variables más influyentes según SHAP son: {", ".join(top_shap)}.
- Los intervalos de confianza de la regresión logística y del Random Forest se traslapan; no hay evidencia de que uno sea mejor.
- Por subgrupos (Random Forest, umbral 0,5): recall de {fmt(rf_mujer["recall"], 2)} en mujeres y de {fmt(rf_mayor["recall"], 2)} en mayores de 65 años; ROC-AUC de {fmt(rf_mayor["roc_auc"], 2)} en mayores de 65 frente a {fmt(rf_menor["roc_auc"], 2)} en menores de 65.
- Calibración: la probabilidad media predicha es {fmt(brier.loc["Random Forest", "prevalencia_media_predicha"], 2)}, frente a una prevalencia observada de {fmt(brier.loc["Random Forest", "prevalencia_observada"], 2)}. Las probabilidades no están calibradas.

## 5. Conclusiones y recomendaciones

1. **Priorizar por riesgo relativo:** el modelo ordena bien a las personas. Un programa que contacte primero al 10 % de mayor riesgo concentra una tasa de evento de {pct(decil1["tasa_evento"])}.
2. **Elegir el umbral según el costo del error:** con umbral 0,5 se detecta cerca de 80 % de los casos, pero 3 de cada 4 alertas son falsas. Un umbral más alto reduce falsas alarmas a costa de detectar menos casos.
3. **Revisar la equidad antes de usar el modelo:** el desempeño cambia entre mujeres y hombres, y entre mayores y menores de 65 años. Debe ajustarse o revisarse por grupo.
4. **No interpretar las probabilidades como riesgo absoluto:** están infladas por el balanceo de clases.
5. **Limitaciones:** datos autoreportados y de corte transversal, sin búsqueda de hiperparámetros y sin validación externa. Los resultados muestran asociaciones, no causas.

## Archivos generados por este cuadernillo

Tablas (`reports/`): {", ".join(archivos_csv)}

Figuras (`reports/figures/`): {", ".join(archivos_fig)}
"""
Path("reports/reporte_ejecutivo.md").write_text(reporte, encoding="utf-8")
display(Markdown(reporte))

## Fin del análisis

Las tablas, las figuras y el reporte ejecutivo quedan en `reports/`. Todos se generan al ejecutar el cuadernillo de principio a fin.